# Field factory — add a new queryable content field with review-only human effort

The Bucket-3 pipeline (detect → sample → label → validate → calibrate → query) existed once,
hardcoded for `alienation_alleged`. This notebook is the **parameterised version**: a user
supplies a field name, a one-paragraph definition and a few seed terms — the factory does
the rest, and the human's only task is **reviewing ~120 drafted labels** (agree/flip), not
creating anything.

Pipeline (all automated except step 5):
1. **Lexicon expansion** — a local LLM expands the seed terms into surface-form patterns
   (printed; editable in the config cell).
2. **Candidate detection** — sentence-level lexicon hits with section context → transparent
   mention-based confidence per case.
3. **Stratified sample** — by confidence band, same design as the alienation gold set
   (all high/mid bands, sampled negatives).
4. **LLM label drafting** — the local LLM applies the definition to each sampled case's
   evidence pool (checkpointed) → `draft_label` + evidence.
5. **Human review** — fill `gold_<field>` in the template (agree or flip). *The only manual
   step.*
6. **Validation** — P/R/F1 for the lexicon detector and the LLM drafts against the reviewed
   gold + out-of-fold isotonic calibration (ECE) → report + deployable calibrated column.

**Stated honesty caveats, by construction:**
- the drafts ARE the LLM extractor's predictions — so the "LLM F1" measured in step 6 is
  agreement with labels the human reviewed *while seeing those drafts* (anchoring). The
  **flip rate** is therefore reported prominently: a near-zero flip rate means the human
  rubber-stamped, and the validation is weak; a substantial flip rate (the alienation
  reference: 37/120 against the rules extractor) means genuine adjudication.
- domain of validity = the corpus this runs on (ECHR-English here), per field, as always.

## 1. Config — the ONLY cell a user edits

In [ ]:
from pathlib import Path

# ==== the three things a user provides ====
FIELD_NAME = "supervised_contact"
FIELD_DEFINITION = (
    "supervised_contact = TRUE if the domestic authorities ordered, arranged or implemented "
    "SUPERVISED contact/visits between a parent and a child - i.e. contact taking place in "
    "the presence of a third person (guardian, psychologist, child-protection officer, "
    "contact centre staff). It is FALSE if supervision was merely requested/discussed but "
    "never ordered, if the contact was unsupervised, or if the term appears only in quoted "
    "legislation. The ORDER/ARRANGEMENT counts even if it later failed in practice.")
SEED_TERMS = ["supervised contact", "supervised visits", "supervised access",
              "in the presence of a third person", "contact centre"]

# ==== knobs (defaults fine) ====
CORPUS_FILE  = Path("../data/echr_parental_alienation.json")   # domain of validity: ECHR-EN
SAMPLE_N     = 120
SEED         = 42
LLM_MODEL    = "llama3.2"
MAX_SENTS    = 10
DATA_DIR     = Path("../data")
REPORT_DIR   = Path("../reports")
TEMPLATE     = DATA_DIR / f"field_{FIELD_NAME}_template.csv"
LABELS       = DATA_DIR / f"field_{FIELD_NAME}_labels.csv"
CKPT         = DATA_DIR / f"field_{FIELD_NAME}_llm_checkpoint.json"
print(f"field: {FIELD_NAME} | corpus: {CORPUS_FILE.name} | sample N={SAMPLE_N}")

## 2. Lexicon expansion (LLM, printed for a human glance)
Seed terms → surface-form patterns. The expansion is *suggested*, printed, and the final
list is whatever `LEXICON` ends up containing — edit here if the LLM proposed junk.

In [ ]:
import json
import re

OLLAMA_OK = False
try:
    import ollama
    ollama.list()
    OLLAMA_OK = True
except Exception as e:
    print(f"Ollama unreachable ({e}) — using seed terms only")

LEXICON = list(SEED_TERMS)
if OLLAMA_OK:
    prompt = (f"List 10 additional short English surface forms (words or phrases, one per "
              f"line, no numbering, no prose) that ECHR judgments use for this concept:\n"
              f"{FIELD_DEFINITION}\nAlready known: {', '.join(SEED_TERMS)}")
    r = ollama.chat(model=LLM_MODEL, messages=[{"role": "user", "content": prompt}],
                    options={"temperature": 0})
    suggested = [l.strip(" -*\u2022").lower() for l in r["message"]["content"].splitlines()
                 if 2 < len(l.strip()) < 60 and not l.strip().endswith(":")]
    print("LLM-suggested additions (review; junk is harmless — it only adds candidates):")
    for s in suggested:
        print("  +", s)
    LEXICON += suggested

LEX_RE = re.compile("|".join(re.escape(t) for t in sorted(set(LEXICON), key=len, reverse=True)),
                    re.IGNORECASE)
print(f"\nfinal lexicon: {len(set(LEXICON))} patterns")

## 3. Candidate detection + transparent confidence
Reuses the deployed ECHR section splitter (exec'd from `echr_extraction.ipynb`). Confidence
is a simple, printable function of mention density and section context — it exists to
*stratify the sample*, not to be the final extractor.

In [ ]:
from pathlib import Path as _P

src_ex = json.loads(_P("echr_extraction.ipynb").read_text())
_ns = {"re": re, "json": json}
for marker in ["ECHR_ANCHORS = ["]:
    cell = next("".join(c["source"]) for c in src_ex["cells"]
                if c["cell_type"] == "code" and marker in "".join(c["source"]))
    exec(cell, _ns)
echr_sections, _SENT = _ns["echr_sections"], _ns["_SENT"]

records = json.loads(CORPUS_FILE.read_text())
print(f"corpus: {len(records)} records")

def detect(full_text):
    """(confidence, evidence pool) — mentions weighted by section context."""
    secs = echr_sections(full_text or "")
    pool, n_law, n_facts = [], 0, 0
    for lab, t in secs:
        for s in _SENT.split(t):
            if LEX_RE.search(s):
                pool.append((lab, s.strip()[:350]))
                if lab == "LAW":
                    n_law += 1
                elif lab in ("FACTS", "HEADER", "PROCEDURE", "unparsed"):
                    n_facts += 1
    if not pool:
        return 0.02, []
    conf = min(1.0, 0.3 + 0.12 * min(n_facts, 3) + 0.08 * min(n_law, 3))
    return round(conf, 3), pool[:MAX_SENTS]

det = {}
for r in records:
    conf, pool = detect(r.get("full_text", ""))
    det[r["itemid"]] = {"conf": conf, "pool": pool,
                        "title": r.get("docname", ""), "genre": r.get("doctype", ""),
                        "url": f"https://hudoc.echr.coe.int/eng?i={r['itemid']}"}
import numpy as np
confs = np.array([d["conf"] for d in det.values()])
print(f"cases with >=1 mention: {(confs > 0.02).sum()} | conf>=0.5: {(confs >= 0.5).sum()}")

## 4. Stratified sample → LLM-drafted template (frozen once)

In [ ]:
import pandas as pd

if LABELS.exists() or TEMPLATE.exists():
    print("template/labels already exist — frozen, not regenerated")
else:
    rng = np.random.default_rng(SEED)
    ids = list(det)
    conf = {i: det[i]["conf"] for i in ids}
    bands = {
        "high(>=.5)":  [i for i in ids if conf[i] >= 0.5],
        "mid[.3,.5)":  [i for i in ids if 0.3 <= conf[i] < 0.5],
        "none(<.3)":   [i for i in ids if conf[i] < 0.3],
    }
    targets = {"high(>=.5)": 60, "mid[.3,.5)": 40, "none(<.3)": 20}
    picks = []
    for name, pool_ids in bands.items():
        k = min(targets[name], len(pool_ids))
        take = pool_ids if k == len(pool_ids) else list(rng.choice(pool_ids, k, replace=False))
        picks += take
        print(f"  {name:12s}: {len(take)}/{len(pool_ids)}")
    picks = picks[:SAMPLE_N]

    # LLM drafts a label per sampled case (checkpointed) — these ARE the extractor's predictions
    PROMPT = ("You classify excerpts from an ECHR family-law case.\n\nDefinition: "
              + FIELD_DEFINITION +
              "\n\nSentences from the case (with document section):\n{sents}\n\n"
              "Reply ONLY with a JSON object: "
              '{{"label": true or false, "confidence": 0.0-1.0, '
              '"evidence": "<best supporting sentence, verbatim>"}}')
    _JSON = re.compile(r"\{.*\}", re.S)
    done = json.loads(CKPT.read_text()) if CKPT.exists() else {}
    todo = [i for i in picks if i not in done]
    print(f"LLM drafting: {len(done)} cached, {len(todo)} to run")
    for n, iid in enumerate(todo, 1):
        pool = det[iid]["pool"]
        if not pool or not OLLAMA_OK:
            done[iid] = {"label": False, "conf": 0.02, "evidence": None}
        else:
            sents = "\n".join(f"- [{lab}] {s}" for lab, s in pool)
            out = None
            for _ in range(2):
                try:
                    r = ollama.chat(model=LLM_MODEL,
                                    messages=[{"role": "user", "content": PROMPT.format(sents=sents)}],
                                    options={"temperature": 0})
                    m = _JSON.search(r["message"]["content"])
                    obj = json.loads(m.group(0))
                    out = {"label": bool(obj.get("label")),
                           "conf": max(0.0, min(1.0, float(obj.get("confidence", 0.5)))),
                           "evidence": str(obj.get("evidence", ""))[:300]}
                    break
                except Exception as exn:
                    err = exn
            done[iid] = out or {"label": False, "conf": 0.5, "evidence": f"unparseable"}
        if n % 10 == 0 or n == len(todo):
            CKPT.write_text(json.dumps(done))
            print(f"  {n}/{len(todo)}")

    rows = [{"id": i, "title": det[i]["title"], "url": det[i]["url"],
             "detector_conf": det[i]["conf"],
             "draft_label": int(done[i]["label"]), "draft_conf": done[i]["conf"],
             "draft_evidence": done[i]["evidence"],
             f"gold_{FIELD_NAME}": "", "notes": ""} for i in picks]
    tmpl = pd.DataFrame(rows).sample(frac=1, random_state=SEED)
    tmpl.to_csv(TEMPLATE, index=False)
    print(f"\nwrote {TEMPLATE.name}: {len(tmpl)} rows | drafted positive: "
          f"{tmpl.draft_label.sum()} ({tmpl.draft_label.mean()*100:.0f}%)")
    print(f"NEXT (the only human step): review gold_{FIELD_NAME} (agree=copy draft, or flip),")
    print(f"save as {LABELS.name}, re-run this notebook.")

## 5. Validation — runs once labels exist (graceful until then)
Reports the **flip rate** first (how often the human overruled the drafts — the integrity
measure), then P/R/F1 of lexicon detector and LLM drafts against gold, then out-of-fold
calibration of the draft confidences.

In [ ]:
if not LABELS.exists():
    print(f"no labels yet — review {TEMPLATE.name}, save as {LABELS.name}, re-run.")
else:
    lab = pd.read_csv(LABELS)
    gcol = f"gold_{FIELD_NAME}"
    lab["gold"] = pd.to_numeric(lab[gcol], errors="coerce")
    lab = lab[lab.gold.notna()].copy()
    lab["gold"] = lab.gold.astype(int)
    flips = int((lab.gold != lab.draft_label).sum())
    print(f"labelled: {len(lab)} | gold positive rate {lab.gold.mean():.3f}")
    print(f"FLIP RATE (human overruled the draft): {flips}/{len(lab)} = "
          f"{flips/len(lab)*100:.0f}%  <- integrity measure; ~0% = rubber-stamp warning")

    from sklearn.metrics import precision_recall_fscore_support
    from sklearn.isotonic import IsotonicRegression
    from sklearn.model_selection import StratifiedKFold
    y = lab.gold.to_numpy()
    for name, pred in [("lexicon detector (conf>=0.5)", (lab.detector_conf >= 0.5).astype(int)),
                       ("LLM drafts", lab.draft_label.astype(int))]:
        p, r, f1, _ = precision_recall_fscore_support(y, pred, average="binary", zero_division=0)
        print(f"  {name:28s}: P={p:.3f} R={r:.3f} F1={f1:.3f}")

    def ece(conf, yy, n_bins=5):
        bins = np.linspace(0, 1, n_bins + 1)
        idx = np.clip(np.digitize(conf, bins) - 1, 0, n_bins - 1)
        return sum((idx == b).sum() / len(yy) * abs(yy[idx == b].mean() - conf[idx == b].mean())
                   for b in range(n_bins) if (idx == b).any())
    raw = lab.draft_conf.to_numpy(float)
    if len(set(y)) > 1 and min((y == 1).sum(), (y == 0).sum()) >= 5:
        oof = np.full_like(raw, np.nan)
        for tr, te in StratifiedKFold(5, shuffle=True, random_state=SEED).split(raw.reshape(-1, 1), y):
            iso = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1)
            iso.fit(raw[tr], y[tr])
            oof[te] = iso.predict(raw[te])
        print(f"  ECE raw={ece(raw, y.astype(float)):.3f} | OOF-calibrated={ece(oof, y.astype(float)):.3f}")

    lines = [f"# Field factory report: `{FIELD_NAME}`\n\n",
             f"- definition: {FIELD_DEFINITION}\n",
             f"- sample: {len(lab)} (stratified), gold positive rate {lab.gold.mean()*100:.0f}%\n",
             f"- **flip rate {flips}/{len(lab)}** (human vs drafts; low = anchoring warning)\n",
             f"- protocol: LLM-drafted labels reviewed by a single human annotator while "
             f"seeing the drafts (anchoring caveat, as with all gold sets in this project); "
             f"domain of validity: {CORPUS_FILE.name} only.\n"]
    (REPORT_DIR / f"field_{FIELD_NAME}_report.md").write_text("".join(lines), encoding="utf-8")
    print("wrote", REPORT_DIR / f"field_{FIELD_NAME}_report.md")
    print(f"\nNEXT: full-corpus LLM pass + calibrated column + query-layer registration "
          f"(same steps as alienation_alleged; run after the validation is acceptable).")